In [2]:
import sys
import pandas as pd
from pathlib import Path
from datetime import datetime

sys.path.append(str(Path('..') / 'src'))
from config import CH

In [3]:
# ══════════════════════════════════════════════
# checks_bronze.py — validaciones Bronze
# ══════════════════════════════════════════════

def check_bronze(ch):
    print("═" * 50)
    print("VALIDACIONES BRONZE")
    print("═" * 50)

    checks = {
        'raw_customers': {
            'total':     "SELECT count() FROM bronze.raw_customers",
            'nulos_pk':  "SELECT count() FROM bronze.raw_customers WHERE customer_id = ''",
            'duplicados':"SELECT count() FROM (SELECT customer_id, count() AS c FROM bronze.raw_customers GROUP BY customer_id HAVING c > 1)",
        },
        'raw_orders': {
            'total':     "SELECT count() FROM bronze.raw_orders",
            'nulos_pk':  "SELECT count() FROM bronze.raw_orders WHERE order_id = 0",
            'sin_fecha': "SELECT count() FROM bronze.raw_orders WHERE order_date IS NULL",
            'duplicados':"SELECT count() FROM (SELECT order_id, count() AS c FROM bronze.raw_orders GROUP BY order_id HAVING c > 1)",
        },
        'raw_order_details': {
            'total':     "SELECT count() FROM bronze.raw_order_details",
            'nulos_pk':  "SELECT count() FROM bronze.raw_order_details WHERE order_id = 0 OR product_id = 0",
            'duplicados':"SELECT count() FROM (SELECT order_id, product_id, count() AS c FROM bronze.raw_order_details GROUP BY order_id, product_id HAVING c > 1)",
        },
        'raw_products': {
            'total':     "SELECT count() FROM bronze.raw_products",
            'nulos_pk':  "SELECT count() FROM bronze.raw_products WHERE product_id = 0",
            'duplicados':"SELECT count() FROM (SELECT product_id, count() AS c FROM bronze.raw_products GROUP BY product_id HAVING c > 1)",
        },
        'raw_categories': {
            'total':     "SELECT count() FROM bronze.raw_categories",
            'nulos_pk':  "SELECT count() FROM bronze.raw_categories WHERE category_id = 0",
            'duplicados':"SELECT count() FROM (SELECT category_id, count() AS c FROM bronze.raw_categories GROUP BY category_id HAVING c > 1)",
        },
        'raw_suppliers': {
            'total':     "SELECT count() FROM bronze.raw_suppliers",
            'nulos_pk':  "SELECT count() FROM bronze.raw_suppliers WHERE supplier_id = 0",
            'duplicados':"SELECT count() FROM (SELECT supplier_id, count() AS c FROM bronze.raw_suppliers GROUP BY supplier_id HAVING c > 1)",
        },
        'raw_employees': {
            'total':     "SELECT count() FROM bronze.raw_employees",
            'nulos_pk':  "SELECT count() FROM bronze.raw_employees WHERE employee_id = 0",
            'duplicados':"SELECT count() FROM (SELECT employee_id, count() AS c FROM bronze.raw_employees GROUP BY employee_id HAVING c > 1)",
        },
        'raw_shippers': {
            'total':     "SELECT count() FROM bronze.raw_shippers",
            'nulos_pk':  "SELECT count() FROM bronze.raw_shippers WHERE shipper_id = 0",
            'duplicados':"SELECT count() FROM (SELECT shipper_id, count() AS c FROM bronze.raw_shippers GROUP BY shipper_id HAVING c > 1)",
        },
        'raw_territories': {
            'total':     "SELECT count() FROM bronze.raw_territories",
            'nulos_pk':  "SELECT count() FROM bronze.raw_territories WHERE territory_id = ''",
            'duplicados':"SELECT count() FROM (SELECT territory_id, count() AS c FROM bronze.raw_territories GROUP BY territory_id HAVING c > 1)",
        },
        'raw_region': {
            'total':     "SELECT count() FROM bronze.raw_region",
            'nulos_pk':  "SELECT count() FROM bronze.raw_region WHERE region_id = 0",
            'duplicados':"SELECT count() FROM (SELECT region_id, count() AS c FROM bronze.raw_region GROUP BY region_id HAVING c > 1)",
        },
    }

    for tabla, validaciones in checks.items():
        print(f"\n── {tabla} ──")
        for nombre, query in validaciones.items():
            resultado = ch.execute(query)[0][0]
            estado    = "OK" if resultado == 0 or nombre == 'total' else "ALERTA"
            simbolo   = "✓" if estado == "OK" else "✗"
            print(f"  {simbolo} {nombre:15} {resultado:>8,}   {estado if estado == 'ALERTA' else ''}")

check_bronze(CH)

══════════════════════════════════════════════════
VALIDACIONES BRONZE
══════════════════════════════════════════════════

── raw_customers ──
  ✓ total                 91   
  ✓ nulos_pk               0   
  ✓ duplicados             0   

── raw_orders ──
  ✓ total                830   
  ✓ nulos_pk               0   
  ✓ sin_fecha              0   
  ✓ duplicados             0   

── raw_order_details ──
  ✓ total              2,155   
  ✓ nulos_pk               0   
  ✓ duplicados             0   

── raw_products ──
  ✓ total                 77   
  ✓ nulos_pk               0   
  ✓ duplicados             0   

── raw_categories ──
  ✓ total                  8   
  ✓ nulos_pk               0   
  ✓ duplicados             0   

── raw_suppliers ──
  ✓ total                 29   
  ✓ nulos_pk               0   
  ✓ duplicados             0   

── raw_employees ──
  ✓ total                  9   
  ✓ nulos_pk               0   
  ✓ duplicados             0   

── raw_shippers ──
  ✓ to

In [4]:
# ══════════════════════════════════════════════
# checks_silver.py — validaciones Silver
# ══════════════════════════════════════════════

def check_silver(ch):
    print("═" * 50)
    print("VALIDACIONES SILVER")
    print("═" * 50)

    checks = {
        'stg_customers': {
            'total':          "SELECT count() FROM silver.stg_customers",
            'nulos_pk':       "SELECT count() FROM silver.stg_customers WHERE customer_id = ''",
            'duplicados':     "SELECT count() FROM (SELECT customer_id, count() AS c FROM silver.stg_customers GROUP BY customer_id HAVING c > 1)",
            'pais_vacio':     "SELECT count() FROM silver.stg_customers WHERE country = ''",
        },
        'stg_orders': {
            'total':          "SELECT count() FROM silver.stg_orders",
            'nulos_pk':       "SELECT count() FROM silver.stg_orders WHERE order_id = 0",
            'duplicados':     "SELECT count() FROM (SELECT order_id, count() AS c FROM silver.stg_orders GROUP BY order_id HAVING c > 1)",
            'sin_customer':   "SELECT count() FROM silver.stg_orders WHERE customer_id = ''",
            'flete_negativo': "SELECT count() FROM silver.stg_orders WHERE freight < 0",
        },
        'stg_order_details': {
            'total':          "SELECT count() FROM silver.stg_order_details",
            'precio_cero':    "SELECT count() FROM silver.stg_order_details WHERE unit_price = 0",
            'cantidad_cero':  "SELECT count() FROM silver.stg_order_details WHERE quantity = 0",
            'descuento_raro': "SELECT count() FROM silver.stg_order_details WHERE discount < 0 OR discount > 1",
        },
        'stg_products': {
            'total':          "SELECT count() FROM silver.stg_products",
            'nulos_pk':       "SELECT count() FROM silver.stg_products WHERE product_id = 0",
            'precio_cero':    "SELECT count() FROM silver.stg_products WHERE unit_price = 0",
            'sin_categoria':  "SELECT count() FROM silver.stg_products WHERE category_id = 0",
        },
        'stg_employees': {
            'total':          "SELECT count() FROM silver.stg_employees",
            'nulos_pk':       "SELECT count() FROM silver.stg_employees WHERE employee_id = 0",
            'sin_nombre':     "SELECT count() FROM silver.stg_employees WHERE full_name = ' '",
        },
        'stg_suppliers': {
            'total':          "SELECT count() FROM silver.stg_suppliers",
            'nulos_pk':       "SELECT count() FROM silver.stg_suppliers WHERE supplier_id = 0",
            'sin_pais':       "SELECT count() FROM silver.stg_suppliers WHERE country = ''",
        },
        'stg_shippers': {
            'total':          "SELECT count() FROM silver.stg_shippers",
            'nulos_pk':       "SELECT count() FROM silver.stg_shippers WHERE shipper_id = 0",
        },
        'stg_territories': {
            'total':          "SELECT count() FROM silver.stg_territories",
            'sin_region':     "SELECT count() FROM silver.stg_territories WHERE region_id = 0",
        },
    }

    for tabla, validaciones in checks.items():
        print(f"\n── {tabla} ──")
        for nombre, query in validaciones.items():
            resultado = ch.execute(query)[0][0]
            estado    = "OK" if resultado == 0 or nombre == 'total' else "ALERTA"
            simbolo   = "✓" if estado == "OK" else "✗"
            print(f"  {simbolo} {nombre:20} {resultado:>8,}   {estado if estado == 'ALERTA' else ''}")

check_silver(CH)

══════════════════════════════════════════════════
VALIDACIONES SILVER
══════════════════════════════════════════════════

── stg_customers ──
  ✓ total                      91   
  ✓ nulos_pk                    0   
  ✓ duplicados                  0   
  ✓ pais_vacio                  0   

── stg_orders ──
  ✓ total                     830   
  ✓ nulos_pk                    0   
  ✓ duplicados                  0   
  ✓ sin_customer                0   
  ✓ flete_negativo              0   

── stg_order_details ──
  ✓ total                   2,155   
  ✓ precio_cero                 0   
  ✓ cantidad_cero               0   
  ✓ descuento_raro              0   

── stg_products ──
  ✓ total                      77   
  ✓ nulos_pk                    0   
  ✓ precio_cero                 0   
  ✓ sin_categoria               0   

── stg_employees ──
  ✓ total                       9   
  ✓ nulos_pk                    0   
  ✓ sin_nombre                  0   

── stg_suppliers ──
  ✓ total   

In [5]:
# ══════════════════════════════════════════════
# checks_gold.py — validaciones Gold
# ══════════════════════════════════════════════

def check_gold(ch):
    print("═" * 50)
    print("VALIDACIONES GOLD")
    print("═" * 50)

    checks = {
        'dim_customers': {
            'total':      "SELECT count() FROM gold.dim_customers",
            'nulos_pk':   "SELECT count() FROM gold.dim_customers WHERE customer_id = ''",
            'duplicados': "SELECT count() FROM (SELECT customer_id, count() AS c FROM gold.dim_customers GROUP BY customer_id HAVING c > 1)",
        },
        'dim_products': {
            'total':           "SELECT count() FROM gold.dim_products",
            'nulos_pk':        "SELECT count() FROM gold.dim_products WHERE product_id = 0",
            'sin_categoria':   "SELECT count() FROM gold.dim_products WHERE category_name = ''",
            'sin_supplier':    "SELECT count() FROM gold.dim_products WHERE supplier_name = ''",
            'precio_cero':     "SELECT count() FROM gold.dim_products WHERE list_price = 0",
        },
        'dim_employees': {
            'total':      "SELECT count() FROM gold.dim_employees",
            'nulos_pk':   "SELECT count() FROM gold.dim_employees WHERE employee_id = 0",
            'sin_nombre': "SELECT count() FROM gold.dim_employees WHERE full_name = ' '",
        },
        'dim_date': {
            'total':      "SELECT count() FROM gold.dim_date",
            'duplicados': "SELECT count() FROM (SELECT date_key, count() AS c FROM gold.dim_date GROUP BY date_key HAVING c > 1)",
        },
        'dim_shippers': {
            'total':    "SELECT count() FROM gold.dim_shippers",
            'nulos_pk': "SELECT count() FROM gold.dim_shippers WHERE shipper_id = 0",
        },
        'fact_sales': {
            'total':           "SELECT count() FROM gold.fact_sales",
            'huerfanos_cust':  "SELECT count() FROM gold.fact_sales fs LEFT JOIN gold.dim_customers dc ON fs.customer_id = dc.customer_id WHERE dc.customer_id = ''",
            'huerfanos_prod':  "SELECT count() FROM gold.fact_sales fs LEFT JOIN gold.dim_products dp ON fs.product_id = dp.product_id WHERE dp.product_id = 0",
            'huerfanos_emp':   "SELECT count() FROM gold.fact_sales fs LEFT JOIN gold.dim_employees de ON fs.employee_id = de.employee_id WHERE de.employee_id = 0",
            'huerfanos_date':  "SELECT count() FROM gold.fact_sales fs LEFT JOIN gold.dim_date dd ON fs.date_key = dd.date_key WHERE dd.date_key = 0",
            'line_total_cero': "SELECT count() FROM gold.fact_sales WHERE line_total = 0",
            'sale_price_cero': "SELECT count() FROM gold.fact_sales WHERE sale_price = 0",
        },
    }

    for tabla, validaciones in checks.items():
        print(f"\n── {tabla} ──")
        for nombre, query in validaciones.items():
            resultado = ch.execute(query)[0][0]
            estado    = "OK" if resultado == 0 or nombre == 'total' else "ALERTA"
            simbolo   = "✓" if estado == "OK" else "✗"
            print(f"  {simbolo} {nombre:25} {resultado:>8,}   {estado if estado == 'ALERTA' else ''}")

check_gold(CH)

══════════════════════════════════════════════════
VALIDACIONES GOLD
══════════════════════════════════════════════════

── dim_customers ──
  ✓ total                           91   
  ✓ nulos_pk                         0   
  ✓ duplicados                       0   

── dim_products ──
  ✓ total                           77   
  ✓ nulos_pk                         0   
  ✓ sin_categoria                    0   
  ✓ sin_supplier                     0   
  ✓ precio_cero                      0   

── dim_employees ──
  ✓ total                            9   
  ✓ nulos_pk                         0   
  ✓ sin_nombre                       0   

── dim_date ──
  ✓ total                       14,975   
  ✓ duplicados                       0   

── dim_shippers ──
  ✓ total                            6   
  ✓ nulos_pk                         0   

── fact_sales ──
  ✓ total                        2,155   
  ✓ huerfanos_cust                   0   
  ✓ huerfanos_prod                   0   
  ✓ huer